# Mini Project — Simple Stock Trading Strategy

## FinTech and Quantitative Trading

### Project Objective

In this mini project, we will build a simple rule-based stock trading strategy using Python.

The strategy uses a **20-day Moving Average (MA20)** to generate simple trading signals.

### Learning Goals

By completing this project, we will learn how to:
1. Obtain historical stock-price data.
2. Explore financial data using Pandas.
3. Calculate daily returns.
4. Calculate a Moving Average.
5. Generate simple BUY/SELL signals.
6. Calculate strategy returns.
7. Compare the strategy with Buy-and-Hold.
8. Visualize price, signals, and performance.

> **Important:** This is an educational project, not investment advice.

## 1. The Basic Idea

The project follows:

**Stock Data → Daily Return → Moving Average → Trading Signal → Strategy Return → Performance Comparison**

### Trading Rule

- If **Close Price > MA20** → hold the stock (`Signal = 1`)
- If **Close Price ≤ MA20** → stay out of the market (`Signal = 0`)

### Key Terms

- **Close Price**: closing price of the stock for a trading day.
- **Return**: percentage change in value.
- **Moving Average**: average price over a selected number of previous periods.
- **Trading Signal**: rule-based indication of whether the strategy is invested.
- **Buy-and-Hold**: buy the stock and keep the position.
- **Volatility**: a measure of return fluctuation.

**Indonesian:** Moving Average = rata-rata bergerak; Trading Signal = sinyal perdagangan; Return = tingkat pengembalian.

## 2. Import Libraries

We use:
- `pandas` for data manipulation.
- `numpy` for numerical operations.
- `matplotlib` for visualization.
- `yfinance` for historical market data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

## 3. Download Stock Data

We use **Samsung Electronics**, ticker `005930.KS`.

The `.KS` suffix identifies the Korea Exchange market in Yahoo Finance.

`yf.download()` returns historical market data such as Open, High, Low, Close, and Volume.

In [ ]:
ticker = "005930.KS"

data = yf.download(
    ticker,
    start="2025-01-01",
    end="2026-01-01",
    auto_adjust=False
)

data.head()

## 4. Basic Data Exploration

Before calculating indicators, inspect the dataset.

This is an important data-analysis habit: understand the data before using it.

In [ ]:
print("Number of rows:", len(data))
print("Number of columns:", len(data.columns))

data.info()

In [ ]:
data.describe()

### Interpretation

`describe()` gives statistics such as mean, standard deviation, minimum, maximum, and percentiles.

The exact values depend on the historical data returned when the notebook is run.

## 5. Calculate Daily Return

Daily return measures the percentage change from one closing price to the next:

\[
R_t = \frac{P_t-P_{t-1}}{P_{t-1}}
\]

Pandas' `.pct_change()` performs this calculation.

The first row becomes `NaN` because there is no previous trading day.

In [ ]:
data["Daily_Return"] = data["Close"].pct_change()

data[["Close", "Daily_Return"]].head(10)

In [ ]:
average_return = data["Daily_Return"].mean()
volatility = data["Daily_Return"].std()

print(f"Average Daily Return: {average_return:.4%}")
print(f"Daily Volatility: {volatility:.4%}")

### Interpretation

- **Average Daily Return** = average daily percentage change.
- **Daily Volatility** = standard deviation of daily returns.

Higher volatility means returns fluctuate more.

## 6. Calculate the 20-Day Moving Average

A 20-day Moving Average is the average closing price over the latest 20 trading observations:

\[
MA_{20,t}=\frac{P_t+P_{t-1}+...+P_{t-19}}{20}
\]

The window moves forward one trading day at a time.

We use it as a simple trend indicator for this educational project.

In [ ]:
data["MA20"] = data["Close"].rolling(window=20).mean()

data[["Close", "MA20"]].tail(10)

The first 19 MA20 values are `NaN` because 20 observations are needed to calculate the first complete 20-day average.

## 7. Generate Trading Signals

### Rule

```text
Close > MA20  → Signal = 1
Close ≤ MA20  → Signal = 0
```

- `1` means the strategy is invested.
- `0` means the strategy is outside the market.

`np.where()` works like a simple IF/ELSE condition.

In [ ]:
data["Signal"] = np.where(
    data["Close"] > data["MA20"],
    1,
    0
)

data[["Close", "MA20", "Signal"]].tail(20)

## 8. Calculate Strategy Return

We use:

\[
Strategy\ Return_t = Signal_{t-1}\times Return_t
\]

The previous signal is used with today's return.

### Why `.shift(1)`?

Using the previous signal helps avoid a simple form of **look-ahead bias**.

**Look-ahead bias** means using information that would not have been available when the trading decision was made.

In [ ]:
data["Strategy_Return"] = (
    data["Signal"].shift(1) * data["Daily_Return"]
)

data[
    ["Close", "MA20", "Signal", "Daily_Return", "Strategy_Return"]
].tail(10)

## 9. Calculate Cumulative Performance

A daily return describes one day. Cumulative performance shows how an initial value would grow over the whole period.

We start with an initial value of `1.0` and compound the returns using `.cumprod()`.

In [ ]:
data["Cumulative_Market"] = (
    1 + data["Daily_Return"].fillna(0)
).cumprod()

data["Cumulative_Strategy"] = (
    1 + data["Strategy_Return"].fillna(0)
).cumprod()

data[
    [
        "Close",
        "Signal",
        "Daily_Return",
        "Strategy_Return",
        "Cumulative_Market",
        "Cumulative_Strategy"
    ]
].tail()

## 10. Compare Buy-and-Hold with the Strategy

**Buy-and-Hold** stays invested throughout the period.

The **MA20 Strategy** is invested only when `Close > MA20`.

A benchmark is useful because a strategy should not be evaluated in isolation.

In [ ]:
market_return = data["Cumulative_Market"].iloc[-1] - 1
strategy_return = data["Cumulative_Strategy"].iloc[-1] - 1

print(f"Buy-and-Hold Return: {market_return:.2%}")
print(f"MA20 Strategy Return: {strategy_return:.2%}")

### How to read the result

If the cumulative value is `1.20`, an initial value of 1.00 became 1.20, corresponding to a 20% cumulative return.

The result applies only to this historical sample. This simple backtest does not include transaction costs, taxes, slippage, liquidity constraints, or other real-world factors.

## 11. Visualize Price and Moving Average

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(data.index, data["Close"], label="Close Price")
plt.plot(data.index, data["MA20"], label="20-Day Moving Average")

plt.title("Samsung Electronics: Close Price and MA20")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid()

plt.show()

The Moving Average should appear smoother than the closing-price series because it averages multiple observations.

## 12. Visualize Strategy Performance

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    data.index,
    data["Cumulative_Market"],
    label="Buy-and-Hold"
)

plt.plot(
    data.index,
    data["Cumulative_Strategy"],
    label="MA20 Strategy"
)

plt.title("Buy-and-Hold vs MA20 Strategy")
plt.xlabel("Date")
plt.ylabel("Growth of Initial Value")
plt.legend()
plt.grid()

plt.show()

## 13. Identify BUY and SELL Signal Changes

A signal change means:

- `0 → 1` = entering the market.
- `1 → 0` = leaving the market.

We can count these changes to understand how often the strategy changes position.

In [ ]:
buy_signals = data[
    (data["Signal"] == 1) &
    (data["Signal"].shift(1) == 0)
]

sell_signals = data[
    (data["Signal"] == 0) &
    (data["Signal"].shift(1) == 1)
]

print("Number of BUY signal changes:", len(buy_signals))
print("Number of SELL signal changes:", len(sell_signals))

## 14. Plot BUY and SELL Signals

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(data.index, data["Close"], label="Close Price")
plt.plot(data.index, data["MA20"], label="MA20")

plt.scatter(
    buy_signals.index,
    buy_signals["Close"],
    marker="^",
    label="BUY"
)

plt.scatter(
    sell_signals.index,
    sell_signals["Close"],
    marker="v",
    label="SELL"
)

plt.title("MA20 Trading Signals")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid()

plt.show()

## 15. Final Data Check

This table connects the main concepts:

**Price → Indicator → Signal → Return → Strategy Performance**

In [ ]:
data[
    [
        "Close",
        "MA20",
        "Signal",
        "Daily_Return",
        "Strategy_Return",
        "Cumulative_Market",
        "Cumulative_Strategy"
    ]
].tail(15)

## 16. Final Conclusion

This project implemented a simple Moving Average trading strategy using historical Samsung Electronics stock data.

The strategy generated an invested position when the closing price was above the 20-day moving average and stayed outside the market otherwise.

The strategy was compared with a Buy-and-Hold benchmark.

### Key Learning Points

1. Financial data can be represented and analyzed using a Pandas DataFrame.
2. Daily returns measure percentage price changes.
3. Moving averages can be calculated with a rolling window.
4. Trading rules can be converted into numerical signals.
5. `.shift(1)` helps prevent a simple form of look-ahead bias.
6. Cumulative returns allow performance to be evaluated over time.
7. A trading strategy should be compared with a benchmark.

### Result

Run the notebook to obtain the exact numerical results:

- Buy-and-Hold Return: **see output above**
- MA20 Strategy Return: **see output above**

This project is intentionally simplified. A real quantitative trading system would require additional considerations such as transaction costs, slippage, liquidity, risk management, position sizing, and out-of-sample testing.